In [6]:
import json
import pandas as pd
import numpy as np
import folium
import pandas as pd

ARCHIVO_JSON = "estaciones_cargame.json"

CSV_SALIDA = "estaciones_cargame_limpio.csv"
PARQUET_SALIDA = "estaciones_cargame_limpio.parquet"

# FUNCIONES

def expandir_diccionarios(df):
    """
    Expande todas las columnas que contienen diccionarios.
    Repite hasta que no existan más niveles anidados.
    """
    while True:
        columnas_dict = []
        for col in df.columns:
            tiene_dict = df[col].dropna().apply(
                lambda x: isinstance(x, dict)
            ).any()
            if tiene_dict:
                columnas_dict.append(col)
        if len(columnas_dict) == 0:
            break
        for col in columnas_dict:
            expandido = pd.json_normalize(df[col])
            expandido.columns = [
                f"{col}_{c}"
                for c in expandido.columns
            ]
            df = pd.concat(
                [
                    df.drop(columns=[col]),
                    expandido
                ],
                axis=1
            )
    return df

def convertir_numericos(df):
    for col in df.columns:
        # Intentar convertir a numérico
        serie = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        porcentaje_validos = (
            serie.notna().sum() /
            max(df[col].notna().sum(), 1)
        )

        if porcentaje_validos > 0.8:
            df[col] = serie

    return df

def limpiar_columnas(df):
    # eliminar columnas totalmente vacías
    df = df.dropna(
        axis=1,
        how="all"
    )
    # eliminar espacios en nombres
    df.columns = (
        df.columns
        .str.strip()
        .str.replace(" ", "_")
        .str.replace("/", "_")
    )
    return df

# CARGAR JSON

with open(
    ARCHIVO_JSON,
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)

# NORMALIZACIÓN INICIAL

df = pd.json_normalize(
    data,
    sep="_"
)

print("=" * 60)
print("Dimensiones iniciales:")
print(df.shape)

# EXPANDIR DICCIONARIOS ANIDADOS

df = expandir_diccionarios(df)

print("=" * 60)
print("Después de expandir diccionarios:")
print(df.shape)

# LIMPIEZA

df = limpiar_columnas(df)

# COORDENADAS GEOGRÁFICAS

for col in df.columns:
    nombre = col.lower()
    if any(
        x in nombre
        for x in [
            "latitud",
            "longitud",
            "latitude",
            "longitude"
        ]
    ):

        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )

# CONVERSIÓN AUTOMÁTICA DE NUMÉRICOS

df = convertir_numericos(df)

# COLUMNAS COMPLETAMENTE DUPLICADAS

df = df.loc[
    :,
    ~df.columns.duplicated()
]

df.head(3)


Dimensiones iniciales:
(214, 79)
Después de expandir diccionarios:
(214, 79)


,Id,Nombre,Direccion,Latitud,Longitud,Altitud,Fechacreacion,Observacion,Acceso,Puntoscarga,...,Precios_Max,Conectores_CCS_Combo_1,Conectores_CHAdeMO,Cobros_Gratuito,Conectores_CCS_Combo_2,Conectores_GB-T_AC,Conectores_Schuko,Conectores_GB-T_DC,Cobros_URV__Kw_consumidos,Cobros_Tiempo_de_carga
0,C5B2C958-AB6A-F111-A079-8187F68C9E81,Estación Cariongo,Carrera 5 con calle 9 esquina centro frente a ...,7.373372,-72.647530,0,2026-06-17,Pueden conocer la tarifa de parquedaero del Ho...,En la Playstore descargar e instalar la App Ce...,1,...,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,63B2466C-0B30-EF11-9B07-005056BD7421,Estación Ocaña,"Calle 7, Av. Francisco Fernández de Contreras ...",8.253773,-73.359350,0,2024-06-21,Los usuarios de vehículos eléctricos o híbrido...,En la Playstore descargar e instalar la App Ce...,1,...,0.0,1.0,1.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN
2,529647BD-E824-ED11-9B91-005056BD1A8D,Estación Unicentro,"Avenida 4 # 2b Norte 81, 12N",7.916802,-72.493302,0,2022-08-25,Los usuarios de vehículos eléctricos o híbrido...,En la Playstore descargar e instalar la App Ce...,1,...,0.0,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
columnas_finales = [
    'Id',
    'Nombre',
    'Direccion',
    'Latitud',
    'Longitud',
    'Fechacreacion',
    'PotenciaMax',
    'Departamento_NombreDepartamento',
    'Municipio_NombreMunicipio',
    'Ubicacion_Nombre'
]

df_final = df[
    [c for c in columnas_finales if c in df.columns]
].copy()

df_final.head(5)

,Id,Nombre,Direccion,Latitud,Longitud,Fechacreacion,PotenciaMax,Departamento_NombreDepartamento,Municipio_NombreMunicipio,Ubicacion_Nombre
0,C5B2C958-AB6A-F111-A079-8187F68C9E81,Estación Cariongo,Carrera 5 con calle 9 esquina centro frente a ...,7.373372,-72.647530,2026-06-17,44.0,NORTE DE SANTANDER,CUCUTA,Hotel
1,63B2466C-0B30-EF11-9B07-005056BD7421,Estación Ocaña,"Calle 7, Av. Francisco Fernández de Contreras ...",8.253773,-73.359350,2024-06-21,50.0,NORTE DE SANTANDER,CUCUTA,Parqueadero
2,529647BD-E824-ED11-9B91-005056BD1A8D,Estación Unicentro,"Avenida 4 # 2b Norte 81, 12N",7.916802,-72.493302,2022-08-25,22.0,NORTE DE SANTANDER,CUCUTA,Centro comercial
3,78BE5448-E838-ED11-9B91-005056BD1A8D,Estación Ventura Plaza,"Calle 10 y 11 con Diagonal Santander, Quinta v...",7.887781,-72.496250,2022-09-20,50.0,NORTE DE SANTANDER,CUCUTA,Centro comercial
4,F6B1389F-7672-EE11-9AFB-005056BD7421,Acapulco,"Virginia – Viterbo, kilómetro 13 (243R+FF)",5.003248,-75.858816,2023-10-24,50.0,CALDAS,MANIZALES,Estación de Servicio


In [8]:
df_final.to_csv(
    'estaciones_cargame_modelo.csv',
    index=False,
    encoding='utf-8-sig'
)

In [9]:
# Mapa

# Eliminar registros sin coordenadas
mapa_df = df_final.dropna(
    subset=['Latitud', 'Longitud']
).copy()

# Centro del mapa (promedio de coordenadas)
centro_lat = mapa_df['Latitud'].mean()
centro_lon = mapa_df['Longitud'].mean()

# Crear mapa
mapa = folium.Map(
    location=[centro_lat, centro_lon],
    zoom_start=6,
    tiles='OpenStreetMap'
)

# Agregar estaciones
for _, row in mapa_df.iterrows():

    popup = f"""
    <b>{row['Nombre']}</b><br>
    Dirección: {row['Direccion']}<br>
    Municipio: {row['Municipio_NombreMunicipio']}<br>
    Departamento: {row['Departamento_NombreDepartamento']}<br>
    Potencia: {row['PotenciaMax']} kW
    """

    folium.Marker(
        [row['Latitud'], row['Longitud']],
        popup=popup
    ).add_to(mapa)

# Guardar mapa
mapa.save("mapa_estaciones.html")

mapa

**Historico de matriculas de vehículos eléctricos**

In [10]:
import pandas as pd

data = {
    "Periodo": [
        "Enero","Febrero","Marzo","Abril","Mayo","Junio",
        "Julio","Agosto","Septiembre","Octubre",
        "Noviembre","Diciembre","Total"
    ],
    "2014":[1,0,0,2,0,0,7,1,4,2,3,0,20],
    "2015":[12,8,2,5,8,41,22,13,29,36,11,6,193],
    "2016":[16,17,9,8,7,18,16,11,18,12,22,48,202],
    "2017":[7,16,26,9,12,7,5,10,16,8,16,4,136],
    "2018":[16,18,26,6,25,35,27,18,32,39,68,80,390],
    "2019":[30,52,40,73,78,47,86,99,139,66,72,144,926],
    "2020":[51,77,110,0,51,69,157,25,82,67,311,307,1307],
    "2021":[45,198,90,37,92,50,79,61,87,148,227,215,1329],
    "2022":[397,323,384,235,323,164,446,154,128,260,232,232,3278],
    "2023":[178,323,316,314,242,246,212,291,318,287,457,493,3677],
    "2024":[217,395,380,363,392,653,778,664,740,1079,1580,1937,9178],
    "2025":[966,1095,1385,1286,1421,1143,1565,1647,1858,2090,2034,3234,19724],
    "2026":[1758,2508,5083,5192,5001,4935,4870,4542,5895,None,None,None,39784]
}

df = pd.DataFrame(data)
display(df)

# Guardar en Excel
#df.to_excel(
#    "Historico_Vehiculos_Electricos.xlsx",
#    index=False
#)
#print("Archivo Excel generado correctamente")

,Periodo,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
0,Enero,1,12,16,7,16,30,51,45,397,178,217,966,1758.0
1,Febrero,0,8,17,16,18,52,77,198,323,323,395,1095,2508.0
2,Marzo,0,2,9,26,26,40,110,90,384,316,380,1385,5083.0
3,Abril,2,5,8,9,6,73,0,37,235,314,363,1286,5192.0
4,Mayo,0,8,7,12,25,78,51,92,323,242,392,1421,5001.0
5,Junio,0,41,18,7,35,47,69,50,164,246,653,1143,4935.0
6,Julio,7,22,16,5,27,86,157,79,446,212,778,1565,4870.0
7,Agosto,1,13,11,10,18,99,25,61,154,291,664,1647,4542.0
8,Septiembre,4,29,18,16,32,139,82,87,128,318,740,1858,5895.0
9,Octubre,2,36,12,8,39,66,67,148,260,287,1079,2090,NaN


**BIBLIOGRAFIA**

De acuerdo con la Resolución MME 40123 de 2024, un CPO (Charge Point Operator) antes de poner en operación la estación de carga se debe registrarla en la plataforma CárgaME,
https://cargame.minenergia.gov.co/#/pages/maps/leaflet

Informe de vehículos eléctricos e Hibridos septiembre 2026
https://www.fenalco.com.co/blog/noticias-10/informe-vehiculos-electricos-e-hibridos-a-septiembre-2026-9195